# TabNet：官方预训练、二分类/回归、调参和模型报告

安装 `hscredit[net,tune]`。`pretrain` 底层调用官方 `TabNetPretrainer`，无监督重建其嵌入空间；
`fit` 使用统一 net 训练接口接收编码器权重。预训练的 `sample_weight` 是官方采样权重，监督训练是损失权重。
本例按时间划分真实数据，独立测试集不参与预训练、早停或选参。少量训练轮次用于验证接口，不代表效果基准。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from numpy.testing import assert_allclose
from sklearn.metrics import roc_auc_score
from openpyxl import load_workbook
from hscredit import TabNet, ModelTuner

root = Path.cwd() if (Path.cwd() / 'examples').is_dir() else Path.cwd().parent
output = root / 'artifacts' / 'tabnet_workflow'
output.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
data = pd.read_excel(root / 'examples' / 'hscredit_yyp.xlsx').sort_values('放款时间', kind='stable')
features = ['衡枢鉴真分老客版', '近六个月非银多头机构数', '青云24', '商品类别']
train = data[data['放款时间'] < '2026-01-14']
valid = data[(data['放款时间'] >= '2026-01-14') & (data['放款时间'] < '2026-02-04')]
test = data[data['放款时间'] >= '2026-02-04']
datasets = {'训练集': train, '选参验证集': valid, '独立测试集': test}
assert [len(frame) for frame in datasets.values()] == [706, 118, 146]
options = dict(n_d=16, n_a=16, n_steps=3, cat_embedding_dim=2, max_epochs=12,
               batch_size=128, virtual_batch_size=64, early_stopping_rounds=4,
               learning_rate=0.01, device='cpu', random_state=42)
print({name: len(frame) for name, frame in datasets.items()})

## 1. 官方半监督预训练的无标签阶段

目标列即使存在也仅移除，不读取。预处理只使用训练集，第一份无标签验证集用于早停。
`pretrainer_` 是官方预训练器，包含官方解码器；`get_embeddings` 返回预测头之前的各决策步表示之和。

In [ ]:
encoder = TabNet(target='FPD', **options)
encoder.pretrain(train[features + ['FPD']], eval_set={'预训练验证集': valid[features]},
                 pretraining_ratio=0.5, max_epochs=5)
embedding = encoder.get_embeddings(test[features])
assert embedding.shape == (len(test), 16) and np.isfinite(embedding).all()
encoder.save(output / '预训练模型.pkl')
restored_encoder = TabNet.load(output / '预训练模型.pkl')
assert_allclose(restored_encoder.transform(test[features]), embedding)
display(encoder.get_learning_curve())

## 2. 二分类、预测概率、结果和原生特征重要性

`from_pretrained` 支持同类已预训练实例；None 自动用本实例快照，False 从头训练。
类别 embedding 的注意力重要性汇总回原始字段，预测概率的两列分别为 0 和 1。

In [ ]:
classifier = TabNet(target='FPD', **options)
classifier.fit(train[features + ['FPD']], eval_set={'选参验证集': valid[features + ['FPD']]},
               from_pretrained=encoder)
probability = classifier.predict_proba(test[features])
assert_allclose(probability.sum(axis=1), 1)
assert_allclose(classifier.predict(test[features]), (probability[:, 1] >= 0.5).astype(int))
assert_allclose(encoder.transform(test[features]), embedding)
display(classifier.get_feature_importances())
print('独立测试集 AUC:', roc_auc_score(test['FPD'], probability[:, 1]))

## 3. 统一调参接口和优化器配置

迁移预训练快照时保持结构一致，只搜索学习率/正则等训练参数。结构搜索可使用未预训练的 TabNet。
自定义 `loss` / `loss_params`、`optimizer` / `optimizer_params` 与其他 net 模型相同。
默认搜索空间使用 n_d、n_a、n_steps 等 TabNet 参数，`AutoTuner.create('tabnet')` 也受支持。

In [ ]:
tuner = ModelTuner(
    TabNet(target='FPD', optimizer='adamw', optimizer_params={'eps': 1e-8}, **options),
    search_space={'learning_rate': [0.005, 0.01], 'lambda_sparse': [0.0001, 0.001]},
    fit_params={'from_pretrained': encoder}, metric='auc', n_jobs=1, random_state=42,
)
tuner.fit(train[features], train['FPD'], eval_set=(valid[features], valid['FPD']),
          n_trials=2, show_progress_bar=False)
best = tuner.get_best_model()
best.save(output / '二分类模型.pkl')
assert_allclose(TabNet.load(output / '二分类模型.pkl').predict_proba(test[features]), best.predict_proba(test[features]))
print('选中的参数:', tuner.best_params_)

## 4. 回归训练与表示输出

相同官方编码器快照可迁移到回归。回归输出恢复到放款金额原始单位，不提供分类概率。

In [ ]:
regressor = TabNet(objective='regression', target='放款金额', loss='huber',
                   loss_params={'delta': 1.0}, **options)
regressor.fit(train[features], train['放款金额'], from_pretrained=encoder,
              eval_set={'选参验证集': (valid[features], valid['放款金额'])})
prediction = regressor.predict(test[features])
assert prediction.shape == (len(test),) and np.isfinite(prediction).all()
assert regressor.transform(test[features]).shape == (len(test), 16)
regressor.save(output / '回归模型.pkl')
assert_allclose(TabNet.load(output / '回归模型.pkl').predict(test[features]), prediction)
print('独立测试集 RMSE:', np.sqrt(np.mean((prediction - test['放款金额'].to_numpy()) ** 2)))

## 5. 分类、回归以及多标签报告

同一个已训练的二分类模型可按 target 映射或 overdue × dpds 重新评估，不重新训练。
SHAP 解释模型输出；注意力重要性和 SHAP 使用统一 TOP32 整合图，依赖图最多 TOP10。
多标签 target 的值用 callable，不能直接填列名字符串。

In [ ]:
explain = dict(enabled=True, scope='importance', dataset='选参验证集',
               max_samples=24, max_background=12, max_evals=2 * len(features) + 1, random_state=42)
reports = {
    '二分类': best.model_report(datasets=datasets, target='FPD', n_jobs=1, verbose=False, explain_config=explain),
    '多标签target': best.model_report(datasets=datasets,
        target={'FPD': lambda d: d['FPD'], 'MOB1@3': lambda d: (d['MOB1'] > 3).astype(int)},
        n_jobs=1, verbose=False, explain_config=explain),
    '逾期多标签': best.model_report(datasets=datasets, overdue=['MOB1'], dpds=[7, 3, 0],
        n_jobs=1, verbose=False, explain_config=explain),
    '回归': regressor.model_report(datasets=datasets, target='放款金额', n_jobs=1, verbose=False, explain_config=explain),
}
records = []
for name, report in reports.items():
    path = output / f'TabNet-{name}报告.xlsx'
    result = report.to_excel(path, amount='放款金额', date='放款时间', group='商品类别',
        loc_cols='客户编号', with_plots=True, mode='strict', return_result=True, model_name='TabNet')
    assert result.complete
    book = load_workbook(path)
    assert '报告执行状态' not in book.sheetnames
    assert all(cell.data_type != 'e' for sheet in book for row in sheet for cell in row)
    assert sum(len(sheet._images) for sheet in book) > 0
    records.append({'报告': name, '路径': str(path), '工作表': book.sheetnames,
                    '图片数': sum(len(sheet._images) for sheet in book)})
    book.close()
    table = report.get_shap_importance()
    explanation = report.get_model_explanation()['解释结果']
    assert_allclose(table.set_index('特征')['平均绝对SHAP值'].reindex(explanation.feature_names),
                    np.abs(explanation.values).mean(axis=0))
    print('已核验:', path)
binary_metrics = reports['二分类'].get_metrics().set_index('统计项')
assert_allclose(binary_metrics.loc['AUC', '独立测试集'], roc_auc_score(test['FPD'], best.predict_proba(test[features])[:, 1]))
regression_metrics = reports['回归'].get_metrics().set_index('统计项')
assert_allclose(regression_metrics.loc['均方根误差', '独立测试集'], np.sqrt(np.mean((prediction-test['放款金额'].to_numpy())**2)))
(output / '验证记录.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(records))

## 6. 原始概率转换为评分后出报告

普通模型报告默认使用 `predict_proba`；纯评分转换器报告默认使用 `transform`。
转换公式、实际参数和采用的推理方法会单独写入模型参数页；`prediction_method` 可以显式覆盖默认值。
逾期标签统一为 `MOB1@7`、`MOB1@3`、`MOB1@0`，比较符及剔灰规则在基本信息中说明。

In [ ]:
from hscredit import ScoreTransformer
from hscredit.report import ModelReport

probability_frames = {
    name: frame.assign(原始概率=best.predict_proba(frame[features])[:, 1])
    for name, frame in datasets.items()
}
converter = ScoreTransformer(method="standard", lower=300, upper=1000, direction="descending")
converter.fit(probability_frames["训练集"]["原始概率"])
score_report = ModelReport(
    converter, datasets=probability_frames, features=["原始概率"],
    overdue=["MOB1"], dpds=[7, 3, 0], overdue_operator=">", del_grey=True, n_jobs=1,
)
assert score_report.prediction_method == "transform"
assert_allclose(score_report._datasets["独立测试集"].prediction,
                converter.transform(probability_frames["独立测试集"]["原始概率"]))
status = score_report.to_excel(output / "TabNet-评分转换报告.xlsx", with_plots=True,
    amount="放款金额", date="放款时间", group="商品类别", mode="strict", return_result=True)
assert status.complete
book = load_workbook(output / "TabNet-评分转换报告.xlsx")
assert any(isinstance(c.value, str) and c.value.endswith("、概率转评分公式")
           for row in book["5-模型参数"] for c in row)
book.close()
print("评分报告:", output / "TabNet-评分转换报告.xlsx")
